In [2]:
# ==========================================================
# IMPORTS
# ==========================================================

import joblib
import numpy as np
import pandas as pd

from pathlib import Path

# ==========================================================
# PROJECT PATHS
# ==========================================================

PROJECT_DIR = Path("..")

DATA_DIR = PROJECT_DIR / "Data"

RESULTS_DIR = PROJECT_DIR / "Results"

RESULTS_DIR.mkdir(exist_ok=True)

# ==========================================================
# LOAD TEST DATA
# ==========================================================

test_df = pd.read_csv(
    DATA_DIR / "test_2023_2025.csv"
)

print(f"Test Dataset Shape : {test_df.shape}")

# ==========================================================
# FEATURES & TARGET
# ==========================================================

TARGET = "Thunderstorm_24h"

DROP_COLUMNS = [
    "Date",
    "Time",
    TARGET
]

X_test = test_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

y_test = test_df[TARGET]

feature_names = X_test.columns.tolist()

print(f"Number of Features : {len(feature_names)}")

# ==========================================================
# LOAD TRAINED MODELS
# ==========================================================

cat_model = joblib.load(
    DATA_DIR / "CatBoost_Final_Research_Model.pkl"
)

lgb_model = joblib.load(
    DATA_DIR / "LightGBM_Thunderstorm_Model.pkl"
)

xgb_model = joblib.load(
    DATA_DIR / "XGBoost_Final.pkl"
)

print("\nModels Loaded Successfully")

print("✓ CatBoost")
print("✓ LightGBM")
print("✓ XGBoost")

Test Dataset Shape : (1392, 28)
Number of Features : 25

Models Loaded Successfully
✓ CatBoost
✓ LightGBM
✓ XGBoost


In [3]:
print(type(lgb_model))
print(type(xgb_model))

<class 'lightgbm.sklearn.LGBMClassifier'>
<class 'xgboost.sklearn.XGBClassifier'>


In [4]:
import pandas as pd
import numpy as np

# ==========================================================
# GENERATE OPERATIONAL FILES
# ==========================================================

def generate_operational_files(
    model,
    X_test,
    model_name,
    feature_importance=None
):
    """
    Creates:
        1. Operational Probability Mapping
        2. Operational Threshold Table
        3. Feature Importance CSV
    """

    # ------------------------------------------------------
    # Predict probabilities
    # ------------------------------------------------------

    probabilities = model.predict_proba(X_test)[:, 1]

    analysis_df = X_test.copy()
    analysis_df["Probability"] = probabilities

    # ======================================================
    # Probability Mapping
    # ======================================================

    _, bin_edges = pd.qcut(
        analysis_df["Probability"],
        q=7,
        retbins=True,
        duplicates="drop"
    )

    probability_mapping = pd.DataFrame({
        "Display_Percentage":[35,45,55,65,75,85,95],
        "Lower_Bound":bin_edges[:-1],
        "Upper_Bound":bin_edges[1:]
    })

    print(f"\n{model_name} Probability Mapping\n")

    display(probability_mapping)

    probability_mapping.to_csv(
        f"{model_name}_Operational_Probability_Mapping.csv",
        index=False
    )

    # ======================================================
    # Probability Levels
    # ======================================================

    analysis_df["Probability Level"] = pd.qcut(

        analysis_df["Probability"],

        q=7,

        labels=[

            "<35% (Very Low)",

            "45% (Low)",

            "55% (Moderately Low)",

            "65% (Moderate)",

            "75% (Moderately High)",

            "85% (High)",

            "95% (Very High)"

        ]

    )

    # ======================================================
    # Features
    # ======================================================

    higher_features = [

        "PW",

        "KI",

        "CAPE_V",

        "CT",

        "SWEAT"

    ]

    lower_features = [

        "LI",

        "SI"

    ]

    order = [

        "95% (Very High)",

        "85% (High)",

        "75% (Moderately High)",

        "65% (Moderate)",

        "55% (Moderately Low)",

        "45% (Low)",

        "<35% (Very Low)"

    ]

    rows=[]

    for level in order:

        subset = analysis_df[
            analysis_df["Probability Level"]==level
        ]

        row={"Probability Level":level}

        for feature in higher_features:

            value=subset[feature].quantile(0.25)

            if feature=="CAPE_V" and value<=1:

                row[feature]="≈0"

            else:

                row[feature]=f"≥ {value:.2f}"

        for feature in lower_features:

            value=subset[feature].quantile(0.75)

            row[feature]=f"≤ {value:.2f}"

        rows.append(row)

    threshold_table=pd.DataFrame(rows)

    print(f"\n{model_name} Operational Thresholds\n")

    display(threshold_table)

    threshold_table.to_csv(

        f"{model_name}_Operational_Thunderstorm_Thresholds.csv",

        index=False

    )

    # ======================================================
    # Feature Importance
    # ======================================================

    if feature_importance is None:

        if hasattr(model,"feature_importances_"):

            importance=model.feature_importances_

        else:

            importance=np.zeros(len(X_test.columns))

    else:

        importance=feature_importance

    importance_df=pd.DataFrame({

        "Feature":X_test.columns,

        "Importance":importance

    }).sort_values(

        "Importance",

        ascending=False

    )

    importance_df.to_csv(

        f"{model_name}_Feature_Importance.csv",

        index=False

    )

    print(f"\n{model_name} files saved successfully.\n")

    return probability_mapping, threshold_table, importance_df

In [5]:
lgb_probability_mapping, \
lgb_thresholds, \
lgb_importance = generate_operational_files(

    model=lgb_model,

    X_test=X_test,

    model_name="LightGBM"
)

[LightGBM] [Warning] feature_fraction is set=0.9207116051626851, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9207116051626851
[LightGBM] [Warning] lambda_l1 is set=4.332378684164573, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.332378684164573
[LightGBM] [Warning] lambda_l2 is set=7.8246451974311855, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.8246451974311855
[LightGBM] [Warning] bagging_fraction is set=0.9076559034865189, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9076559034865189
[LightGBM] [Warning] bagging_freq is set=2, subsample_freq=0 will be ignored. Current value: bagging_freq=2

LightGBM Probability Mapping



,Display_Percentage,Lower_Bound,Upper_Bound
0,35,0.001123,0.005160
1,45,0.005160,0.020233
2,55,0.020233,0.110538
3,65,0.110538,0.288888
4,75,0.288888,0.481831
5,85,0.481831,0.682527
6,95,0.682527,0.960100



LightGBM Operational Thresholds



,Probability Level,PW,KI,CAPE_V,CT,SWEAT,LI,SI
0,95% (Very High),≥ 59.69,≥ 35.30,≥ 2313.39,≥ 19.60,≥ 222.00,≤ -3.39,≤ -0.13
1,85% (High),≥ 55.95,≥ 34.10,≥ 2113.62,≥ 18.70,≥ 217.70,≤ -3.22,≤ 0.56
2,75% (Moderately High),≥ 48.57,≥ 31.50,≥ 1709.35,≥ 17.50,≥ 203.85,≤ -3.01,≤ 1.21
3,65% (Moderate),≥ 41.96,≥ 26.40,≥ 1667.89,≥ 15.65,≥ 156.15,≤ -2.38,≤ 3.10
4,55% (Moderately Low),≥ 31.21,≥ 18.15,≥ 915.28,≥ 11.35,≥ 101.35,≤ -0.95,≤ 6.39
5,45% (Low),≥ 23.27,≥ -11.85,≥ 30.22,≥ 7.30,≥ 62.20,≤ 1.77,≤ 10.44
6,<35% (Very Low),≥ 17.56,≥ -25.45,≈0,≥ -5.85,≥ 40.25,≤ 6.36,≤ 15.87



LightGBM files saved successfully.



In [6]:
xgb_probability_mapping, \
xgb_thresholds, \
xgb_importance = generate_operational_files(

    model=xgb_model,

    X_test=X_test,

    model_name="XGBoost"
)



XGBoost Probability Mapping



,Display_Percentage,Lower_Bound,Upper_Bound
0,35,0.002134,0.007821
1,45,0.007821,0.035188
2,55,0.035188,0.154816
3,65,0.154816,0.361661
4,75,0.361661,0.555175
5,85,0.555175,0.721117
6,95,0.721117,0.934408



XGBoost Operational Thresholds



,Probability Level,PW,KI,CAPE_V,CT,SWEAT,LI,SI
0,95% (Very High),≥ 59.80,≥ 35.48,≥ 2259.97,≥ 19.70,≥ 223.55,≤ -3.61,≤ -0.39
1,85% (High),≥ 56.44,≥ 34.30,≥ 2159.62,≥ 18.50,≥ 220.70,≤ -3.27,≤ 0.57
2,75% (Moderately High),≥ 49.14,≥ 31.38,≥ 2014.11,≥ 17.85,≥ 203.95,≤ -2.95,≤ 0.95
3,65% (Moderate),≥ 41.84,≥ 26.62,≥ 1422.01,≥ 15.35,≥ 161.75,≤ -2.37,≤ 3.04
4,55% (Moderately Low),≥ 31.54,≥ 17.84,≥ 974.34,≥ 11.20,≥ 101.10,≤ -1.23,≤ 6.57
5,45% (Low),≥ 23.65,≥ -10.90,≥ 86.63,≥ 6.90,≥ 57.80,≤ 1.02,≤ 9.97
6,<35% (Very Low),≥ 17.26,≥ -27.40,≈0,≥ -8.90,≥ 38.00,≤ 6.39,≤ 16.24



XGBoost files saved successfully.



In [7]:
import pandas as pd
import numpy as np

# ==========================================================
# ENSEMBLE OPERATIONAL FILES
# ==========================================================

# ----------------------------------------------------------
# Ensemble Probability (Average of 3 Models)
# ----------------------------------------------------------

cat_prob = cat_model.predict_proba(X_test)[:, 1]
lgb_prob = lgb_model.predict_proba(X_test)[:, 1]
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

ensemble_prob = (
    cat_prob +
    lgb_prob +
    xgb_prob
) / 3

analysis_df = X_test.copy()
analysis_df["Probability"] = ensemble_prob

# ==========================================================
# SAVE OPERATIONAL PROBABILITY MAPPING
# ==========================================================

_, bin_edges = pd.qcut(
    analysis_df["Probability"],
    q=7,
    retbins=True,
    duplicates="drop"
)

probability_mapping = pd.DataFrame({
    "Display_Percentage":[35,45,55,65,75,85,95],
    "Lower_Bound":bin_edges[:-1],
    "Upper_Bound":bin_edges[1:]
})

print("\nEnsemble Operational Probability Mapping\n")
display(probability_mapping)

probability_mapping.to_csv(
    "Ensemble_Operational_Probability_Mapping.csv",
    index=False
)

# ==========================================================
# PROBABILITY LEVELS
# ==========================================================

analysis_df["Probability Level"] = pd.qcut(
    analysis_df["Probability"],
    q=7,
    labels=[
        "<35% (Very Low)",
        "45% (Low)",
        "55% (Moderately Low)",
        "65% (Moderate)",
        "75% (Moderately High)",
        "85% (High)",
        "95% (Very High)"
    ]
)

# ==========================================================
# THRESHOLD TABLE
# ==========================================================

higher_features = [
    "PW",
    "KI",
    "CAPE_V",
    "CT",
    "SWEAT"
]

lower_features = [
    "LI",
    "SI"
]

order = [
    "95% (Very High)",
    "85% (High)",
    "75% (Moderately High)",
    "65% (Moderate)",
    "55% (Moderately Low)",
    "45% (Low)",
    "<35% (Very Low)"
]

rows = []

for level in order:

    subset = analysis_df[
        analysis_df["Probability Level"] == level
    ]

    row = {
        "Probability Level": level
    }

    for feature in higher_features:

        value = subset[feature].quantile(0.25)

        if feature == "CAPE_V" and value <= 1:
            row[feature] = "≈0"
        else:
            row[feature] = f"≥ {value:.2f}"

    for feature in lower_features:

        value = subset[feature].quantile(0.75)

        row[feature] = f"≤ {value:.2f}"

    rows.append(row)

threshold_table = pd.DataFrame(rows)

print("\nEnsemble Operational Thunderstorm Thresholds\n")
display(threshold_table)

threshold_table.to_csv(
    "Ensemble_Operational_Thunderstorm_Thresholds.csv",
    index=False
)

# ==========================================================
# ENSEMBLE FEATURE IMPORTANCE
# ==========================================================

ensemble_importance = (
    cat_model.feature_importances_
    + lgb_model.feature_importances_
    + xgb_model.feature_importances_
) / 3

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": ensemble_importance
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

print("\nEnsemble Feature Importance\n")
display(importance_df)

importance_df.to_csv(
    "Ensemble_Feature_Importance.csv",
    index=False
)

print("\nAll Ensemble files saved successfully!")

[LightGBM] [Warning] feature_fraction is set=0.9207116051626851, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9207116051626851
[LightGBM] [Warning] lambda_l1 is set=4.332378684164573, reg_alpha=0.0 will be ignored. Current value: lambda_l1=4.332378684164573
[LightGBM] [Warning] lambda_l2 is set=7.8246451974311855, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.8246451974311855
[LightGBM] [Warning] bagging_fraction is set=0.9076559034865189, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9076559034865189
[LightGBM] [Warning] bagging_freq is set=2, subsample_freq=0 will be ignored. Current value: bagging_freq=2

Ensemble Operational Probability Mapping



,Display_Percentage,Lower_Bound,Upper_Bound
0,35,0.002638,0.007201
1,45,0.007201,0.030116
2,55,0.030116,0.121166
3,65,0.121166,0.277702
4,75,0.277702,0.439287
5,85,0.439287,0.602714
6,95,0.602714,0.880231



Ensemble Operational Thunderstorm Thresholds



,Probability Level,PW,KI,CAPE_V,CT,SWEAT,LI,SI
0,95% (Very High),≥ 60.07,≥ 35.30,≥ 2332.47,≥ 19.70,≥ 223.35,≤ -3.55,≤ -0.25
1,85% (High),≥ 55.53,≥ 34.30,≥ 2090.49,≥ 18.60,≥ 217.10,≤ -3.30,≤ 0.55
2,75% (Moderately High),≥ 50.13,≥ 31.65,≥ 1871.34,≥ 17.88,≥ 206.00,≤ -2.73,≤ 0.99
3,65% (Moderate),≥ 41.78,≥ 26.90,≥ 1305.05,≥ 15.55,≥ 156.85,≤ -2.37,≤ 2.93
4,55% (Moderately Low),≥ 31.12,≥ 16.40,≥ 1133.53,≥ 11.40,≥ 99.20,≤ -1.47,≤ 6.34
5,45% (Low),≥ 23.30,≥ -8.90,≥ 56.39,≥ 6.90,≥ 59.80,≤ 1.14,≤ 9.97
6,<35% (Very Low),≥ 17.31,≥ -26.90,≈0,≥ -7.80,≥ 40.00,≤ 6.36,≤ 16.12



Ensemble Feature Importance



,Feature,Importance
19,LCL_P,1587.074072
22,MML_MR,1471.233055
21,MML_PT,1360.268742
3,SWEAT,1321.236823
24,PW,1295.004164
4,KI,1232.833190
15,LFC_V,1230.056049
14,LFC,1228.740163
16,BRN,1111.703923
18,LCL_T,1008.665534



All Ensemble files saved successfully!


In [8]:
import pandas as pd
import numpy as np

# ==========================================================
# NORMALIZE FEATURE IMPORTANCES TO 100%
# ==========================================================

def save_normalized_feature_importance(model, X_test, model_name):

    # Get raw importance
    importance = model.feature_importances_.astype(float)

    # Normalize to percentage
    importance = importance / importance.sum() * 100

    importance_df = pd.DataFrame({
        "Feature": X_test.columns,
        "Importance (%)": importance
    })

    importance_df = importance_df.sort_values(
        "Importance (%)",
        ascending=False
    ).reset_index(drop=True)

    print(f"\n{model_name} Feature Importance (%)\n")
    display(importance_df)

    importance_df.to_csv(
        f"{model_name}_Feature_Importance.csv",
        index=False
    )

    print(f"{model_name} Feature Importance saved successfully!")

    return importance_df

In [9]:
lgb_importance_df = save_normalized_feature_importance(
    lgb_model,
    X_test,
    "LightGBM"
)


LightGBM Feature Importance (%)



,Feature,Importance (%)
0,LCL_P,6.411397
1,MML_MR,5.945064
2,MML_PT,5.488166
3,SWEAT,5.335867
4,PW,5.202437
5,KI,4.976009
6,LFC_V,4.970618
7,LFC,4.962532
8,BRN,4.493504
9,LCL_T,4.071648


LightGBM Feature Importance saved successfully!


In [12]:
import joblib
import pandas as pd

# Normalized importance
importance = lgb_model.feature_importances_.astype(float)
importance = importance / importance.sum() * 100

importance_df = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

# Dictionary used by Streamlit
feature_importance = dict(
    zip(
        importance_df["Feature"],
        importance_df["Importance"]
    )
)

# Feature order used by predict.py
feature_order = list(X_test.columns)

joblib.dump(
    feature_importance,
    "LightGBM_feature_importance.pkl"
)

joblib.dump(
    feature_order,
    "LightGBM_feature_order.pkl"
)

print("LightGBM PKL files saved successfully!")

LightGBM PKL files saved successfully!


In [13]:
import joblib
import pandas as pd

importance = xgb_model.feature_importances_.astype(float)
importance = importance / importance.sum() * 100

importance_df = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": importance
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance = dict(
    zip(
        importance_df["Feature"],
        importance_df["Importance"]
    )
)

feature_order = list(X_test.columns)

joblib.dump(
    feature_importance,
    "XGBoost_feature_importance.pkl"
)

joblib.dump(
    feature_order,
    "XGBoost_feature_order.pkl"
)

print("XGBoost PKL files saved successfully!")

XGBoost PKL files saved successfully!


In [14]:
import joblib
import pandas as pd

cat_imp = cat_model.feature_importances_.astype(float)
cat_imp = cat_imp / cat_imp.sum()

lgb_imp = lgb_model.feature_importances_.astype(float)
lgb_imp = lgb_imp / lgb_imp.sum()

xgb_imp = xgb_model.feature_importances_.astype(float)
xgb_imp = xgb_imp / xgb_imp.sum()

ensemble_imp = (cat_imp + lgb_imp + xgb_imp) / 3
ensemble_imp = ensemble_imp * 100

importance_df = (
    pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": ensemble_imp
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance = dict(
    zip(
        importance_df["Feature"],
        importance_df["Importance"]
    )
)

feature_order = list(X_test.columns)

joblib.dump(
    feature_importance,
    "Ensemble_feature_importance.pkl"
)

joblib.dump(
    feature_order,
    "Ensemble_feature_order.pkl"
)

print("Ensemble PKL files saved successfully!")

Ensemble PKL files saved successfully!
